In [1]:
import os
import sys
import math
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

In [67]:
class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads):
        super().__init__()
        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.head_dim = emb_dim // num_heads

        self.Q = nn.Linear(self.emb_dim, self.num_heads * self.head_dim)
        self.K = nn.Linear(self.emb_dim, self.num_heads * self.head_dim)
        self.V = nn.Linear(self.emb_dim, self.num_heads * self.head_dim)
        self.output = nn.Linear(self.num_heads * self.head_dim, self.emb_dim)

    def forward(self, query, key, value, mask=None):
        batch_size, seq_len, _ = query.shape
        Q_out = self.Q(query) # (batch_size, seq_len, num_heads * head_dim)
        K_out = self.K(key)
        V_out = self.V(value)

        # reshaping for multi-head attention

        queries = Q_out.view(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        keys = K_out.view(batch_size, key.shape[1], self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        values = V_out.view(batch_size, value.shape[1], self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        keys_out = torch.matmul(queries, keys.transpose(-2, -1)) # (batch_size, num_heads, seq_len, seq_len)

        if mask is not None:
            keys_out = keys_out.masked_fill(mask == 0, -1e20) 

        scaled_out = F.softmax(keys_out / math.sqrt(self.head_dim), dim=-1) # softmax applied to each row
        scaled_out = torch.matmul(scaled_out, values) # back to (batch_size, num_heads, seq_len, head_dim)
        # concat heads back to its initial dim
        scaled_out = scaled_out.permute(0, 2, 1, 3).contiguous().view(batch_size, seq_len, -1) # flatten last two dim (batch_size, seq_len, num_heads * head_dim)
        output_o = self.output(scaled_out) # (batch_size, seq_len, emb_dim)

        return output_o

In [68]:
class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, num_heads, dropout, forward_dim):
        super().__init__()
        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.dropout = nn.Dropout(dropout)
        self.forward_dim = forward_dim
        self.layer_norm_1 = nn.LayerNorm(self.emb_dim, eps = 1e-6)
        self.layer_norm_2 = nn.LayerNorm(self.emb_dim, eps = 1e-6)
        self.ff = nn.Sequential(
            nn.Linear(emb_dim, forward_dim),
            nn.ReLU(),
            nn.Linear(forward_dim, emb_dim)
        )
        self.MHA = MultiHeadAttention(emb_dim = emb_dim, num_heads = num_heads)

    def forward(self, query, key, value, mask):
        attention = self.MHA(query, key, value, mask = None)
        attention_skip_con = attention + query
        attention_dropout = self.dropout(attention_skip_con)
        attention_normalized = self.layer_norm_1(attention_dropout)
        ff_output = self.ff(attention_normalized)
        ff_skip_con = ff_output + attention_normalized
        ff_dropout = self.dropout(ff_skip_con)
        output_norm = self.layer_norm_2(ff_dropout)
        return output_norm

In [69]:
# positional encodings
def get_sinusoid_table(max_len, emb_dim):
    def get_angle(pos, i, emb_dim):
        return pos / 10000 ** ((2 * (i // 2)) / emb_dim)

    sinusoid_table = torch.zeros(max_len, emb_dim) # +1 for "<PAD>" at 0
    for pos in range(max_len):
        for i in range(emb_dim):
            if i % 2 == 0:
                sinusoid_table[pos, i] = math.sin(get_angle(pos, i, emb_dim))
            else:
                sinusoid_table[pos, i] = math.cos(get_angle(pos, i, emb_dim))
    return sinusoid_table

In [70]:
class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        emb_dim,
        num_layers,
        num_heads,
        forward_dim,
        dropout,
        max_len,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.forward_dim = forward_dim
        self.dropout = dropout
        self.max_len = max_len

        self.embedding = nn.Embedding(
            num_embeddings = self.vocab_size,
            embedding_dim = self.emb_dim
        )

        self.pos_embedding = nn.Embedding.from_pretrained(
            get_sinusoid_table(self.max_len + 1, self.emb_dim), freeze=True
        )

        self.dropout_layer = nn.Dropout(self.dropout)
        self.transformer_blocks = nn.ModuleList(
            [
                TransformerBlock(self.emb_dim, self.num_heads, self.dropout, self.forward_dim)
                for i in range(self.num_layers)
            ]
        ) # connecting transformer layers to each other (1 block = attention + ffn)

    def forward(self, x, mask):
        batch_size, seq_len = x.shape
        dev = x.device
        token_emb = self.embedding(x)
        # adding positional embeddings
        i_tokens = torch.arange(1, seq_len + 1, device = dev).expand(batch_size, seq_len) # adding indices for each token in the sentence
        pos_emb = self.pos_embedding(i_tokens)
        embeddings = token_emb + pos_emb
        dropout_emb = self.dropout_layer(embeddings)

        input_x = dropout_emb # first input
        for layer in self.transformer_blocks:
            input_x = layer(input_x, input_x, input_x, mask)

        return input_x


In [71]:
class DecoderBlock(nn.Module):
    def __init__(self, emb_dim, num_heads, forward_dim, dropout):
        super().__init__()
        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.forward_dim = forward_dim
        self.dropout = dropout

        self.layer_norm = nn.LayerNorm(self.emb_dim)
        self.MHA_module = MultiHeadAttention(self.emb_dim, self.num_heads) # self-attention
        self.Transformer = TransformerBlock(self.emb_dim, self.num_heads, self.dropout, self.forward_dim) # cross-attention + FFN
        self.dropout_layer = nn.Dropout(self.dropout)

    def forward(self, x, value, key, src_mask, tgt_mask):
        self_attention = self.MHA_module(x, x, x, tgt_mask)
        query = self_attention + x
        query = self.dropout_layer(query)
        query = self.layer_norm(query)
        cross_attention = self.Transformer(
            query, key, value, src_mask
        )

        return cross_attention

In [72]:
class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        emb_dim,
        num_layers,
        num_heads,
        forward_dim,
        dropout,
        max_len
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.forward_dim = forward_dim
        self.max_len = max_len
        self.dropout = dropout

        # Initializing token embeddings for decoder
        self.embeddings = nn.Embedding(
            num_embeddings=self.vocab_size,
            embedding_dim=self.emb_dim
        )

        self.pos_emb = nn.Embedding.from_pretrained(
            get_sinusoid_table(self.max_len + 1, self.emb_dim), freeze=True
            )

        self.dropout_layer = nn.Dropout(self.dropout)
        self.decoder_blocks = nn.ModuleList(
            [
                DecoderBlock(self.emb_dim, self.num_heads, self.forward_dim, self.dropout)
                for i in range(self.num_layers)
            ]
        )

        # final linear mapping to vocab size
        self.linear_output = nn.Linear(self.emb_dim, self.vocab_size)

    def forward(self, x, encoder_out, src_mask, tgt_mask):
        _, seq_len = x.shape
        embeddings = self.embeddings(x)
        i_tokens = torch.arange(seq_len, device = x.device).unsqueeze(0)
        pos_emb = self.pos_emb(i_tokens)
        embeddings = embeddings + pos_emb
        input_decoder = self.dropout_layer(embeddings)

        for layer in self.decoder_blocks:
            input_decoder = layer(input_decoder, encoder_out, encoder_out, src_mask, tgt_mask)

        output = self.linear_output(input_decoder) # softmax maybe?

        return output


In [73]:
class DecoderBlock(nn.Module):
    def __init__(self, emb_dim, num_heads, forward_dim, dropout):
        super().__init__()
        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.forward_dim = forward_dim
        self.dropout = dropout
        
        self.layer_norm = nn.LayerNorm(self.emb_dim, eps=1e-6)
        self.MHA_module = MultiHeadAttention(self.emb_dim, self.num_heads)
        self.Transformer = TransformerBlock(self.emb_dim, self.num_heads, self.dropout, self.forward_dim)
        self.dropout_layer = nn.Dropout(self.dropout)

    def forward(self, x, value, key, src_mask, tgt_mask):
        self_attention = self.MHA_module(x, x, x, tgt_mask)
        query = self_attention + x
        query = self.dropout_layer(query)
        query = self.layer_norm(query)
        cross_attention = self.Transformer(
            query, key, value, src_mask # changed
        )

        return cross_attention

class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        emb_dim,
        num_layers,
        num_heads,
        forward_dim,
        dropout,
        max_len
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.forward_dim = forward_dim
        self.max_len = max_len
        self.dropout = dropout

        # we initialize token embeddings, a dropout layer, num_layers x DecoderBlocks 
        self.embeddings = nn.Embedding(
            num_embeddings=self.vocab_size,
            embedding_dim=self.emb_dim
        )
        self.dropout_layer = nn.Dropout(self.dropout)
        self.decoder_blocks = nn.ModuleList(
            [
                DecoderBlock(self.emb_dim, self.num_heads, self.forward_dim, self.dropout)
                for i in range(self.num_layers)
            ]
        )
        # inside another module list
        # We also need pos encodings, but here we use relative pos encodings
        self.relative_embeddings = nn.Embedding(
            num_embeddings = self.max_len,
            embedding_dim=self.emb_dim
        )

        self.linear_output = nn.Linear(self.emb_dim, self.vocab_size)

    def forward(self, x, encoder_out, src_mask, tgt_mask):
        _, seq_len = x.shape
        embeddings = self.embeddings(x)
        # creates inputs to the relative positional encodings by again creating a matrix of position 
        # indices from each token in the sequence (no `+1` shifting this time because 
        # we train each position relative to the current encoded sequence position output)
        i_tokens = torch.arange(seq_len, device = x.device).unsqueeze(0)
        pos_emb = self.relative_embeddings(i_tokens)
        embeddings = embeddings + pos_emb
        input_decoder = self.dropout_layer(embeddings)

        for layer in self.decoder_blocks:
            input_decoder = layer(input_decoder, encoder_out, encoder_out, src_mask, tgt_mask)

        output = self.linear_output(input_decoder)

        return output

In [76]:
class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        src_pad_idx,
        tgt_pad_idx,
        emb_dim=512,
        num_layers=6,
        num_heads=8,
        forward_dim=2048,
        dropout=0.0,
        max_len=128,
    ):
        super().__init__()

        self.encoder = Encoder(
            vocab_size=src_vocab_size,
            emb_dim=emb_dim,
            num_layers=num_layers,
            num_heads=num_heads,
            forward_dim=forward_dim,
            dropout=dropout,
            max_len=max_len
        )

        self.decoder = Decoder(
            vocab_size=tgt_vocab_size,
            emb_dim=emb_dim,
            num_layers=num_layers,
            num_heads=num_heads,
            forward_dim=forward_dim,
            dropout=dropout,
            max_len=max_len
        )

        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx
        self.num_heads = num_heads

    def create_src_mask(self, src):
        device = src.device
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
        return src_mask.to(device)

    def create_tgt_mask(self, tgt):
        device = tgt.device
        batch_size, tgt_len = tgt.shape
        tgt_mask = (tgt != self.tgt_pad_idx).unsqueeze(1).unsqueeze(2)
        tgt_mask = tgt_mask * torch.tril(torch.ones((tgt_len, tgt_len))).expand(
            batch_size, 1, tgt_len, tgt_len
        ).to(device)
        return tgt_mask

    def forward(self, src, tgt):
        mask_src = self.create_src_mask(src)
        mask_tgt = self.create_tgt_mask(tgt)
        encoded_seq = self.encoder(src, mask_src)
        decoded_seq = self.decoder(tgt, encoded_seq, mask_src, mask_tgt)
        print(f'Encoder shape: {encoded_seq.shape}')
        print(f'Decoder shape: {decoded_seq.shape}')

        return decoded_seq

In [77]:
# general test case
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = Transformer(
    src_vocab_size=200,
    tgt_vocab_size=220,
    src_pad_idx=0,
    tgt_pad_idx=0,
).to(device)

# source input: batch size 4, sequence length of 75
src_in = torch.randint(0, 200, (4, 75)).to(device)

# target input: batch size 4, sequence length of 80
tgt_in = torch.randint(0, 220, (4, 80)).to(device)

# expected output shape of the model
expected_out_shape = torch.Size([4, 80, 220])

with torch.no_grad():
    out = model(src_in, tgt_in)

assert out.shape == expected_out_shape, f"wrong output shape, expected: {expected_out_shape}"

Encoder shape: torch.Size([4, 75, 512])
Decoder shape: torch.Size([4, 80, 220])
